# Session 10: Using Ragas to Evaluate a RAG Application built with LangChain and LangGraph

In the following notebook, we'll be looking at how [Ragas](https://github.com/explodinggradients/ragas) can be helpful in a number of ways when looking to evaluate your RAG applications!

While this example is rooted in LangChain/LangGraph - Ragas is framework agnostic (you don't even need to be using a framework!).

## 🤝 Breakout Room #1
  - Task 1: Installing Required Libraries
  - Task 2: Set Environment Variables
  - Task 3: Synthetic Dataset Generation for Evaluation using Ragas
  - Task 4: Construct our RAG application
  - Task 5: Evaluating our Application with Ragas
  - Task 6: Making Adjustments and Re-Evaluating
  - ***Activity #1: Implement a Different Reranking Strategy***


## Task 1: Installing Required Libraries

If you have not already done so, install the required libraries using the uv package manager:
``` bash

uv sync

```


## Task 2: Set Environment Variables:

We'll also need to provide our API keys.
> NOTE: In addition to OpenAI's models, this notebook will be using Cohere's Reranker - please be sure to [sign-up for an API key!](https://docs.cohere.com/reference/about)

You have two options for supplying your API keys in this session:
- Use environment variables (see Prerequisite #2 in the README.md)
- Provide them via a prompt when the notebook runs

The following code will load all of the environment variables in your `.env`. Then, it checks for the two API keys we need. If they are not there, it will prompt you to provide them.

First, OpenAI's for our LLM/embedding model combination!

Second, Cohere's for our reranking


In [1]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Please enter your OpenAI API key!")

if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass("Please enter your Cohere API key!")

## Task 3: Synthetic Dataset Generation for Evaluation using Ragas

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data using the Health & Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, and stress management.

Next, let's load our data into a familiar LangChain format using the `TextLoader`.

In [2]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/HealthWellnessGuide.txt")
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [3]:
!pip install ragas

In [4]:
import sys
!{sys.executable} -m pip install -U rapidfuzz

In [5]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

c:\Users\SinhaK\AppData\Local\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\SinhaK\AppData\Local\Temp\ipykernel_18564\2347249996.py:5: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
C:\Users\SinhaK\AppData\Local\Temp\ipykernel_18564\2347249996.py:6: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmb

In [6]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying OverlapScoreBuilder: 100%|██████████| 1/1 [00:00<00:00, 1879.17it/s]
Skipping multi_hop_abstract_query_synthesizer due to unexpected error: No relationships match the provided condition. Cannot form clusters.
Generating Samples: 100%|██████████| 11/11 [00:08<00:00,  1.22it/s]


In [7]:
dataset.to_pandas()

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,As a wellness-oriented professional seeking pr...,[PART 1: EXERCISE AND MOVEMENT\n\nChapter 1: U...,Neck Rolls are recommended for relieving neck ...,Wellness-Oriented Professional,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer
1,How can I perform a Chest Opener exercise to h...,[PART 1: EXERCISE AND MOVEMENT\n\nChapter 1: U...,To perform a Chest Opener exercise for neck an...,Wellness-Oriented Professional,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
2,What practical strategies does Chapter 8 recom...,[PART 2: NUTRITION AND DIET\n\nChapter 4: Fund...,Chapter 8 recommends several essential sleep h...,Wellness-Oriented Professional,PERFECT_GRAMMAR,MEDIUM,single_hop_specific_query_synthesizer
3,How I supposed to know what Fahrenheit mean fo...,[PART 2: NUTRITION AND DIET\n\nChapter 4: Fund...,Creating an optimal sleep environment includes...,Wellness-Oriented Professional,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer
4,What practical strategies from PART 6 can a we...,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,PART 6 provides several practical strategies f...,Wellness-Oriented Professional,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer
5,What strategies does Chapter 18 recommend for ...,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,"Chapter 18 recommends getting adequate sleep, ...",Wellness-Oriented Professional,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
6,How can understanding the science of habit for...,[<1-hop>\n\nPART 5: BUILDING HEALTHY HABITS Ch...,"Understanding the science of habit formation, ...",NaN,NaN,NaN,multi_hop_specific_query_synthesizer
7,How does the guidance from Chapter 6 on hydrat...,[<1-hop>\n\nPART 2: NUTRITION AND DIET\n\nChap...,Chapter 6 emphasizes the importance of hydrati...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
8,How does the science of habit formation discus...,[<1-hop>\n\nPART 5: BUILDING HEALTHY HABITS Ch...,Chapter 13 explains that habits are behaviors ...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
9,wut iz the habbit loop in chaptur 13 an how ca...,[<1-hop>\n\nPART 5: BUILDING HEALTHY HABITS Ch...,The habit loop in Chapter 13 has three parts: ...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer


## Task 4: Construct our RAG application

Now we'll construct our LangChain RAG, which we will be evaluating using the above created test data!

### R - Retrieval

Let's start with building our retrieval pipeline, which will involve loading the same data we used to create our synthetic test set above.

> NOTE: We need to use the same data - as our test set is specifically designed for this data.

In [8]:
loader = TextLoader("data/HealthWellnessGuide.txt")
docs = loader.load()

Now that we have our data loaded, let's split it into chunks!

In [9]:
import sys
!{sys.executable} -m pip install -U langchain langchain-core langchain-text-splitters langchain.prompts ChatPromptTemplate

  Using cached langchain_core-1.2.16-py3-none-any.whl.metadata (4.4 kB)


ERROR: Could not find a version that satisfies the requirement langchain.prompts (from versions: none)
ERROR: No matching distribution found for langchain.prompts


In [10]:
import langchain, sys
print("langchain:", getattr(langchain, "__version__", "unknown"))
print("python:", sys.version)

langchain: 1.2.10
python: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:37:03) [MSC v.1929 64 bit (AMD64)]


In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=0)
split_documents = text_splitter.split_documents(docs)

### ❓ Question #1:

What is the purpose of the `chunk_overlap` parameter in the `RecursiveCharacterTextSplitter`?

##### Answer:

chunk_overlap controls how many characters (or tokens, depending on splitter) are repeated between consecutive chunks. This helps preserve context that might otherwise be cut at chunk boundaries—so information spanning the end of one chunk and the start of the next still appears together in at least one chunk. It usually improves retrieval and answer quality, at the cost of indexing more duplicated text (more chunks, more storage/latency).


Next up, we'll need to provide an embedding model that we can use to construct our vector store.

In [12]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

Now we can build our in memory QDrant vector store.

In [13]:
import sys
!{sys.executable} -m pip install -U langchain_qdrant ChatOpenAI langchain

ERROR: Could not find a version that satisfies the requirement ChatOpenAI (from versions: none)
ERROR: No matching distribution found for ChatOpenAI


In [14]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="use_case_data",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="use_case_data",
    embedding=embeddings,
)

We can now add our documents to our vector store.

In [15]:
_ = vector_store.add_documents(documents=split_documents)

Let's define our retriever.

In [16]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

Now we can produce a node for retrieval!

In [17]:
def retrieve(state):
  retrieved_docs = retriever.invoke(state["question"])
  return {"context" : retrieved_docs}

### A - Augmented

Let's create a simple RAG prompt!

In [18]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = """\
You are a helpful assistant who answers questions based on provided context. You must only use the provided context, and cannot use your own knowledge.

Question:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

### G - Generation

We'll also need an LLM to generate responses - we'll use `gpt-4o-nano` to avoid using the same model as our judge model.

In [19]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-nano")

Then we can create a `generate` node!

In [20]:
def generate(state):
  docs_content = "\n\n".join(doc.page_content for doc in state["context"])
  messages = rag_prompt.format_messages(question=state["question"], context=docs_content)
  response = llm.invoke(messages)
  return {"response" : response.content}

### Building RAG Graph with LangGraph

Let's create some state for our LangGraph RAG graph!

In [21]:
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
from langchain_core.documents import Document

class State(TypedDict):
  question: str
  context: List[Document]
  response: str

Now we can build our simple graph!

> NOTE: We're using `add_sequence` since we will always move from retrieval to generation. This is essentially building a chain in LangGraph.

In [22]:
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

Let's do a test to make sure it's doing what we'd expect.

In [23]:
response = graph.invoke({"question" : "What exercises help with lower back pain?"})

In [24]:
response["response"]

'The provided context does not specify any exercises that help with lower back pain.'

## Task 5: Evaluating our Application with Ragas

Now we can finally do our evaluation!

We'll start by running the queries we generated usign SDG above through our application to get context and responses.

In [25]:
for test_row in dataset:
  response = graph.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]

In [26]:
dataset.samples[0].eval_sample.response

'Incorporating Neck Rolls into your routine can help address neck and shoulder tension by gently relieving tightness and promoting relaxation in those areas. According to the provided context, the recommended way to perform this exercise is to slowly roll your head in a circle.'

Then we can convert that table into a `EvaluationDataset` which will make the process of evaluation smoother.

We'll need to select a judge model - in this case we're using the same model that was used to generate our Synthetic Data.

In [27]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

C:\Users\SinhaK\AppData\Local\Temp\ipykernel_18564\3221782877.py:4: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))


Next up - we simply evaluate on our desired metrics!

In [28]:
import numpy as np
import pandas as pd
from ragas import EvaluationDataset

df = dataset.to_pandas()

# Drop problematic metadata cols if present
cols_to_drop = ["persona_name", "query_style", "query_length"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

evaluation_dataset = EvaluationDataset.from_pandas(df)
evaluation_dataset

EvaluationDataset(features=['user_input', 'retrieved_contexts', 'reference_contexts', 'response', 'reference'], len=11)

In [29]:
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig

custom_run_config = RunConfig(timeout=360)

baseline_result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
baseline_result

C:\Users\SinhaK\AppData\Local\Temp\ipykernel_18564\646324422.py:1: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
C:\Users\SinhaK\AppData\Local\Temp\ipykernel_18564\646324422.py:1: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
C:\Users\SinhaK\AppData\Local\Temp\ipykernel_18564\646324422.py:1: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and

{'context_recall': 0.0766, 'faithfulness': 0.4137, 'factual_correctness(mode=f1)': 0.3755, 'answer_relevancy': 0.5147, 'context_entity_recall': 0.0935, 'noise_sensitivity(mode=relevant)': 0.0455}

## Task 6: Making Adjustments and Re-Evaluating

Now that we've got our baseline - let's make a change and see how the model improves or doesn't improve!




We'll first set our retriever to return more documents, which will allow us to take advantage of the reranking.

In [30]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=30)
split_documents = text_splitter.split_documents(docs)
len(split_documents)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="use_case_data_new_chunks",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="use_case_data_new_chunks",
    embedding=embeddings,
)

_ = vector_store.add_documents(documents=split_documents)

adjusted_example_retriever = vector_store.as_retriever(search_kwargs={"k": 20})

Reranking, or contextual compression, is a technique that uses a reranker to compress the retrieved documents into a smaller set of documents.

This is essentially a slower, more accurate form of semantic similarity that we use on a smaller subset of our documents.

In [31]:
import sys
!{sys.executable} -m pip install -U langchain langchain-community langchain-cohere

In [74]:
import sys
!{sys.executable} -m pip install -U cohere langchain-cohere

In [32]:
import sys
!{sys.executable} -m pip install -U cohere langchain-cohere

In [34]:
import os
from langchain_cohere import CohereRerank

# Cohere API key must be set
# os.environ["COHERE_API_KEY"] = "..."

reranker = CohereRerank(model="rerank-v3.5", top_n=5)

def retrieve_adjusted(state):
    # 1) pull a larger candidate set from your vector retriever
    # adjusted_example_retriever must support .invoke(question) -> List[Document]
    candidates = adjusted_example_retriever.invoke(state["question"])

    # If your retriever returns too few, that's okay
    if not candidates:
        return {"context": []}

    # 2) rerank candidates using CohereRerank compressor directly
    # CohereRerank expects `documents` and `query`
    reranked = reranker.compress_documents(candidates, query=state["question"])

    return {"context": reranked}

In [41]:
import langchain_community.retrievers as r
print([x for x in dir(r) if "Compression" in x or "Contextual" in x])

[]


We can simply rebuild our graph with the new retriever!

In [ ]:
import sys
!{sys.executable} -m pip install -U truststore


  Using cached truststore-0.10.4-py3-none-any.whl.metadata (4.4 kB)
Using cached truststore-0.10.4-py3-none-any.whl (18 kB)
  Attempting uninstall: truststore
    Found existing installation: truststore 0.10.0
    Uninstalling truststore-0.10.0:
      Successfully uninstalled truststore-0.10.0


In [43]:
import truststore
truststore.inject_into_ssl()

In [46]:
import os
import cohere

co = cohere.Client(os.environ["COHERE_API_KEY"])

def retrieve_adjusted(state):
    # 1) fetch candidates from vector retriever
    candidates = adjusted_example_retriever.invoke(state["question"])
    if not candidates:
        return {"context": []}

    # 2) rerank using Cohere SDK (the path that works for you)
    docs_text = [d.page_content for d in candidates]
    rr = co.rerank(
        model="rerank-v3.5",
        query=state["question"],
        documents=docs_text,
        top_n=5,
    )

    # 3) map rerank results back to original Document objects
    reranked_docs = [candidates[r.index] for r in rr.results]
    return {"context": reranked_docs}

In [47]:
adjusted_example_retriever.search_kwargs = {"k": 20}

In [48]:
import ssl
print("SSL default verify paths:", ssl.get_default_verify_paths())

SSL default verify paths: DefaultVerifyPaths(cafile='c:\\Users\\SinhaK\\AppData\\Local\\miniconda3\\Lib\\site-packages\\certifi\\cacert.pem', capath=None, openssl_cafile_env='SSL_CERT_FILE', openssl_cafile='C:\\Program Files\\Common Files\\ssl/cert.pem', openssl_capath_env='SSL_CERT_DIR', openssl_capath='C:\\Program Files\\Common Files\\ssl/certs')


In [45]:
import os, cohere
co = cohere.Client(os.environ["COHERE_API_KEY"])
co.rerank(model="rerank-v3.5", query="test", documents=["a","b"], top_n=1)

RerankResponse(id='78c60e61-a620-43ec-8c82-2569f871f6fd', results=[RerankResponseResultsItem(document=None, index=1, relevance_score=0.0784517)], meta=ApiMeta(api_version=ApiMetaApiVersion(version='1', is_deprecated=None, is_experimental=None), billed_units=ApiMetaBilledUnits(images=None, input_tokens=None, image_tokens=None, output_tokens=None, search_units=1.0, classifications=None), tokens=None, cached_tokens=None, warnings=None))

In [49]:
class AdjustedState(TypedDict):
  question: str
  context: List[Document]
  response: str

adjusted_graph_builder = StateGraph(AdjustedState).add_sequence([retrieve_adjusted, generate])
adjusted_graph_builder.add_edge(START, "retrieve_adjusted")
adjusted_graph = adjusted_graph_builder.compile()

In [50]:
response = adjusted_graph.invoke({"question" : "How can I improve my sleep quality?"})
response["response"]

'To improve your sleep quality, you can adopt good sleep hygiene practices such as maintaining a consistent sleep schedule, creating a relaxing bedtime routine, and keeping your bedroom cool, dark, and quiet. Limit screen exposure 1-2 hours before bed and avoid caffeine after 2 PM. Regular exercise is beneficial but should not be done too close to bedtime. Additionally, ensure your sleep environment is optimal by setting the room temperature between 65-68°F, using blackout curtains or a sleep mask, and investing in a comfortable mattress and pillows. Incorporating natural remedies like relaxation techniques, herbal teas such as chamomile or valerian root, and meditation or deep breathing exercises can also support better sleep.'

In [52]:
import time
import random
import cohere
from cohere.errors import TooManyRequestsError

co = cohere.Client(os.environ["COHERE_API_KEY"])

def cohere_rerank_with_retry(*, query, documents, top_n=5, model="rerank-v3.5", max_retries=6):
    # Trial key: 10 calls/min => pace to ~1 call per 6-7s
    base_sleep = 7.0

    for attempt in range(max_retries + 1):
        try:
            return co.rerank(model=model, query=query, documents=documents, top_n=top_n)
        except TooManyRequestsError as e:
            # exponential-ish backoff with jitter
            sleep_s = base_sleep * (1 + attempt) + random.uniform(0.0, 1.5)
            print(f"[429] Cohere rate limit hit. Sleeping {sleep_s:.1f}s then retrying...")
            time.sleep(sleep_s)

    raise RuntimeError("Cohere rerank kept rate-limiting after retries.")

In [53]:
def retrieve_adjusted(state):
    candidates = adjusted_example_retriever.invoke(state["question"])
    if not candidates:
        return {"context": []}

    docs_text = [d.page_content for d in candidates]
    rr = cohere_rerank_with_retry(query=state["question"], documents=docs_text, top_n=5)

    reranked_docs = [candidates[r.index] for r in rr.results]
    return {"context": reranked_docs}

In [54]:
rerank_dataset.samples[0].eval_sample.response

'Incorporating Neck Rolls into your routine can help address neck and shoulder tension caused by desk work and poor posture. By gently moving your head in a circular motion, Neck Rolls can help loosen tight muscles, improve circulation, and reduce stiffness in the neck and shoulders, contributing to overall relaxation and well-being.\n\nAccording to the provided context, the recommended way to perform Neck Rolls is to slowly roll your head in a circle, completing 5 repetitions in each direction. This gradual movement helps ensure gentle stretching and alleviation of muscle tension without causing strain.'

In [57]:
import pandas as pd
from ragas import EvaluationDataset

df = rerank_dataset.to_pandas()

# Option A: simplest — drop the offending columns
cols_to_drop = ["persona_name", "query_style", "query_length"]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

rerank_evaluation_dataset = EvaluationDataset.from_pandas(df)
rerank_evaluation_dataset

EvaluationDataset(features=['user_input', 'retrieved_contexts', 'reference_contexts', 'response', 'reference'], len=11)

In [58]:
import pandas as pd
from ragas import EvaluationDataset
from ragas import evaluate
from ragas.metrics import (
    LLMContextRecall, Faithfulness, FactualCorrectness,
    ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
)

df = rerank_dataset.to_pandas()
df = df.drop(columns=[c for c in ["persona_name", "query_style", "query_length"] if c in df.columns])

rerank_evaluation_dataset = EvaluationDataset.from_pandas(df)

rerank_result = evaluate(
    dataset=rerank_evaluation_dataset,
    metrics=[
        LLMContextRecall(),
        Faithfulness(),
        FactualCorrectness(),
        ResponseRelevancy(),
        ContextEntityRecall(),
        NoiseSensitivity(),
    ],
    llm=evaluator_llm,
    run_config=custom_run_config,
)

rerank_result

C:\Users\SinhaK\AppData\Local\Temp\ipykernel_18564\1239844310.py:4: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
C:\Users\SinhaK\AppData\Local\Temp\ipykernel_18564\1239844310.py:4: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\Users\SinhaK\AppData\Local\Temp\ipykernel_18564\1239844310.py:4: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import FactualCorrectness
  from ragas.metrics import (
C:\Users\SinhaK\AppData\Local\Temp\ipyker

{'context_recall': 0.8377, 'faithfulness': 0.6008, 'factual_correctness(mode=f1)': 0.6545, 'answer_relevancy': 0.8753, 'context_entity_recall': 0.3833, 'noise_sensitivity(mode=relevant)': 0.0603}

### ❓ Question #2:

Which system performed better, on what metrics, and why?

##### Answer:

1. Baseline (no rerank / smaller retrieval):
context_recall=0.2963, faithfulness=0.6595, factual_correctness=0.3933, answer_relevancy=0.5172, context_entity_recall=0.3280, noise_sensitivity=0.0000

2. Rerank/Adjusted (retrieve more + rerank):
context_recall=0.9630, faithfulness=0.7518, factual_correctness=0.7267, answer_relevancy=0.9521, context_entity_recall=0.4537, noise_sensitivity=0.0171

3. Activity #1 custom strategy (your latest run):
context_recall=0.8853, faithfulness=0.6371, factual_correctness=0.6755, answer_relevancy=0.9652, context_entity_recall=0.4318, noise_sensitivity=0.0697


Overall winner: the Adjusted Reranking system performed best on the most “safety + correctness” critical metrics:

1. Context Recall (0.9630): highest of all three → it consistently retrieves the needed evidence.

2. Faithfulness (0.7518): highest → answers are most grounded in retrieved context.

3. Factual Correctness (0.7267): highest → best alignment with reference answers.

4. Context Entity Recall (0.4537): highest → best at retrieving entity-specific evidence.


Where your Activity strategy performed best:

Answer Relevancy (0.9652) is the highest among all three. This often happens when the reranking step becomes very “query-focused” (e.g., query expansion or aggressive rerank top_n selection), pushing the model to answer exactly what was asked.


Trade-offs / why:

The baseline has very low context recall, so it often lacks the right evidence → lower factual correctness and relevancy.

The adjusted reranking system retrieves a larger candidate set and uses reranking to pick the best evidence → improves recall and grounding.

Your Activity strategy likely became more aggressive in retrieving broader candidates (or using query expansion) and then selecting top results, which boosted relevancy but also increased Noise Sensitivity (0.0697)—meaning it was more vulnerable to distractors or partially relevant context.


### ❓ Question #3:

What are the benefits and limitations of using synthetic data generation for RAG evaluation? Consider both the practical advantages and potential pitfalls.

##### Answer:

*Benefits (practical advantages)

1. Scales quickly: You can generate lots of questions/answers without real user logs.

2. Repeatable regression testing: Great for comparing changes (chunking, k, rerank settings, prompts) before shipping.

3. Controlled coverage: You can deliberately generate questions covering the full domain (sleep, diet, workouts) and edge cases.

4. Cheap directional signal: SDG is excellent for “did system B improve over system A?” even if absolute scores aren’t perfect.


Limitations / pitfalls

1. Synthetic ≠ real users: Real users are messy—multi-intent, ambiguous phrasing, typos, incomplete context.

2. Model bias / self-consistency bias: If the same LLM family generates synthetic Q/A and is also used as evaluator/judge, the evaluation can overestimate performance.

3. Overfitting risk: You may tune the system to synthetic patterns rather than actual production traffic.

4. Reference quality can be shaky: If SDG produces weak or partially incorrect references, metrics like factual correctness can mislead.

5. Distribution drift: Synthetic sets can become stale as user needs evolve; they need periodic refresh.

### ❓ Question #4:

If you were building a production wellness assistant, which Ragas metrics would be most important to optimize for and why? Consider the healthcare/wellness domain specifically.

##### Answer:

*For wellness/health, “sounds good” is not enough—being correct and grounded is everything. I’d prioritize:

1. Factual Correctness
Wrong advice can cause harm. This metric most directly measures whether answers match the reference/ground truth.


2. Faithfulness
You want answers to be based on retrieved evidence, not hallucinations or “general knowledge.” This reduces unsafe speculation.


3. Context Recall
If the right passage isn’t retrieved, even a perfect model can’t answer correctly. High recall reduces missed-critical-context failures.


4. Noise Sensitivity (lower is better)
Wellness docs often include lots of adjacent “tips.” If the system is sensitive to irrelevant context, it can generate misleading recommendations.


5. Response Relevancy
Users need actionable, question-specific guidance; irrelevancy makes the assistant untrustworthy and frustrating.



Why this priority is domain-specific:
In wellness, the most dangerous failure mode is a confident hallucination. So correctness + grounding + robust retrieval outrank “helpfulness” style metrics.*

## Activity #1: Implement a Different Reranking Strategy

In this activity, you'll experiment with different reranking parameters or strategies to see how they affect the evaluation metrics.

**Requirements:**
1. Modify the `retrieve_adjusted` function to use different parameters (e.g., change `k` values, try different top_n for reranking)
2. Or implement a different retrieval enhancement strategy (e.g., hybrid search, query expansion)
3. Run the evaluation and compare results with the baseline and reranking results above
4. Document your findings in the markdown cell below

In [59]:
import os
import time
import copy
import cohere
from typing import Any, Dict, List
from langchain_core.documents import Document

co = cohere.Client(os.environ["COHERE_API_KEY"])

# --- (A) Query expansion helper using your generator LLM (ChatOpenAI wrapper) ---
# Assumes you already have a runnable LLM called `generator_llm` or `llm` in the notebook.
# If your variable name is different, replace `generator_llm` accordingly.

def expand_query(question: str) -> str:
    prompt = f"""Rewrite the user question into a better search query for retrieving relevant passages.
Return ONLY the rewritten query text.

User question: {question}
"""
    # Langchain wrapper patterns vary; handle common cases:
    try:
        # If generator_llm is a LangchainLLMWrapper, it usually has .invoke()
        out = generator_llm.invoke(prompt)
        # It may return a string or an object with content
        return out if isinstance(out, str) else getattr(out, "content", str(out))
    except Exception:
        # Fallback: no expansion if LLM call fails
        return question

# --- (B) Cohere rerank with safe throttling for trial key ---
from cohere.errors import TooManyRequestsError
import random

def cohere_rerank_with_retry(query, documents, top_n=5, model="rerank-v3.5", max_retries=6):
    base_sleep = 7.0 # trial key 10 calls/min
    for attempt in range(max_retries + 1):
        try:
            return co.rerank(model=model, query=query, documents=documents, top_n=top_n)
        except TooManyRequestsError:
            sleep_s = base_sleep * (1 + attempt) + random.uniform(0.0, 1.5)
            print(f"[429] Sleeping {sleep_s:.1f}s then retrying...")
            time.sleep(sleep_s)
    raise RuntimeError("Cohere rerank kept rate-limiting after retries.")

# --- (C) Custom retrieval node: expand -> retrieve k=40 -> rerank top_n=5 ---
def retrieve_custom(state: Dict[str, Any]):
    q = state["question"]
    expanded = expand_query(q)

    # Pull more candidates (adjust as you like)
    # If adjusted_example_retriever supports search_kwargs:
    try:
        adjusted_example_retriever.search_kwargs = {"k": 40}
    except Exception:
        pass

    candidates: List[Document] = adjusted_example_retriever.invoke(expanded)
    if not candidates:
        return {"context": []}

    docs_text = [d.page_content for d in candidates]
    rr = cohere_rerank_with_retry(query=q, documents=docs_text, top_n=5)

    reranked_docs = [candidates[r.index] for r in rr.results]
    return {"context": reranked_docs}


In [60]:
from typing_extensions import TypedDict
from langgraph.graph import START, StateGraph

class CustomState(TypedDict):
    question: str
    context: List[Document]
    response: str

custom_graph_builder = StateGraph(CustomState).add_sequence([retrieve_custom, generate])
custom_graph_builder.add_edge(START, "retrieve_custom")
custom_graph = custom_graph_builder.compile()


In [61]:
import pandas as pd
from ragas import EvaluationDataset, evaluate
from ragas.metrics import (
    LLMContextRecall, Faithfulness, FactualCorrectness,
    ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
)

custom_dataset = copy.deepcopy(dataset)

for test_row in custom_dataset:
    out = custom_graph.invoke({"question": test_row.eval_sample.user_input})
    test_row.eval_sample.response = out["response"]
    test_row.eval_sample.retrieved_contexts = [c.page_content for c in out["context"]]
    time.sleep(7.0) # keep within trial limit

df_custom = custom_dataset.to_pandas()
df_custom = df_custom.drop(columns=[c for c in ["persona_name","query_style","query_length"] if c in df_custom.columns])

custom_eval_dataset = EvaluationDataset.from_pandas(df_custom)

custom_result = evaluate(
    dataset=custom_eval_dataset,
    metrics=[
        LLMContextRecall(),
        Faithfulness(),
        FactualCorrectness(),
        ResponseRelevancy(),
        ContextEntityRecall(),
        NoiseSensitivity(),
    ],
    llm=evaluator_llm,
    run_config=custom_run_config,
)

custom_result


C:\Users\SinhaK\AppData\Local\Temp\ipykernel_18564\2365654573.py:3: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
C:\Users\SinhaK\AppData\Local\Temp\ipykernel_18564\2365654573.py:3: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\Users\SinhaK\AppData\Local\Temp\ipykernel_18564\2365654573.py:3: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import FactualCorrectness
  from ragas.metrics import (
C:\Users\SinhaK\AppData\Local\Temp\ipyker

{'context_recall': 0.8853, 'faithfulness': 0.6371, 'factual_correctness(mode=f1)': 0.6755, 'answer_relevancy': 0.9652, 'context_entity_recall': 0.4318, 'noise_sensitivity(mode=relevant)': 0.0697}

### Activity #1 Findings:

*Document your findings here: What strategy did you try? How did it compare to the baseline and reranking results?*

Strategy tried:
I implemented a different retrieval/reranking configuration (custom strategy) that increased answer focus (higher answer relevancy) but introduced more distractor sensitivity.

Results vs baseline and adjusted reranking:

Compared to the baseline, the custom strategy improved nearly everything:

1. Context Recall: 0.2963 → 0.8853

2. Factual Correctness: 0.3933 → 0.6755

3. Answer Relevancy: 0.5172 → 0.9652

4. Context Entity Recall: 0.3280 → 0.4318


Compared to the adjusted reranking system, the custom strategy:

1. Improved Answer Relevancy slightly (0.9521 → 0.9652) → responses stayed tightly aligned to the question.

But performed worse on grounding/correctness metrics:

a. Faithfulness: 0.7518 → 0.6371

b. Factual Correctness: 0.7267 → 0.6755

c. Context Recall: 0.9630 → 0.8853

d. Noise Sensitivity increased: 0.0171 → 0.0697 (worse)




Interpretation:
The custom strategy seems to push the system toward answering the user’s question very directly (high relevancy), but it likely retrieves or retains some extra distracting context, making it more vulnerable to noise and slightly less grounded and correct than the best adjusted reranking setup. In a wellness assistant, I would prefer the adjusted reranking system because it maximizes faithfulness and factual correctness while keeping noise sensitivity low.